In [ ]:
# ==============================
# INSTALL (run once in Colab)
# ==============================
# !pip install transformers datasets peft accelerate scikit-learn

# ==============================
# IMPORTS
# ==============================
from google.colab import drive
drive.mount('/content/drive')
import time
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding # Added this import
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score

# ==============================
# CONFIG
# ==============================
MODEL_NAME = "bert-base-uncased"
DATASET = "imdb"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 2


device = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# LOAD DATA
# ==============================
dataset = load_dataset(DATASET)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

# ==============================
# METRICS
# ==============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# ==============================
# TRAIN FUNCTION
# ==============================
def train_model(model, tag="baseline"):
    model.to(device)

    args = TrainingArguments(
        output_dir=f"/content/drive/MyDrive/results_{tag}",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=50,
        load_best_model_at_end=True,
        fp16=True,
        report_to="none"
    )

    # Explicitly define data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    start_time = time.time()
    try:
        trainer.train(resume_from_checkpoint=True)
    except Exception as e:
        print(f"Could not resume from checkpoint: {e}. Starting new training run.")
        trainer.train()
    end_time = time.time()

    results = trainer.evaluate()
    training_time = end_time - start_time

    return results, training_time

# ==============================
# PARAM COUNT
# ==============================
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ==============================
# 1. FULL FINE-TUNING
# ==============================
print("\n===== FULL FINE-TUNING =====")

model_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

params_full = count_trainable_params(model_full)

results_full, time_full = train_model(model_full, "full")

# ==============================
# 2. LoRA FINE-TUNING
# ==============================
print("\n===== LoRA FINE-TUNING =====")

model_lora = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model_lora = get_peft_model(model_lora, lora_config)

params_lora = count_trainable_params(model_lora)

results_lora, time_lora = train_model(model_lora, "lora")

# ==============================
# FINAL COMPARISON
# ==============================
print("\n===== FINAL RESULTS =====")

print("\nFULL FINE-TUNING")
print(results_full)
print(f"Training Time: {time_full:.2f} sec")
print(f"Trainable Params: {params_full}")

print("\nLoRA FINE-TUNING")
print(results_lora)
print(f"Training Time: {time_lora:.2f} sec")
print(f"Trainable Params: {params_lora}")

Mounted at /content/drive
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Epoch,Training Loss,Validation Loss,Accuracy,F1
2,0.355940,0.378348,0.843000,0.846829


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



===== FINAL RESULTS =====

FULL FINE-TUNING
{'eval_loss': 0.30819982290267944, 'eval_accuracy': 0.888, 'eval_f1': 0.8864097363083164, 'eval_runtime': 900.2805, 'eval_samples_per_second': 1.111, 'eval_steps_per_second': 0.139, 'epoch': 2.0}
Training Time: 78.05 sec
Trainable Params: 109483778

LoRA FINE-TUNING
{'eval_loss': 0.3783477544784546, 'eval_accuracy': 0.843, 'eval_f1': 0.8468292682926829, 'eval_runtime': 882.2505, 'eval_samples_per_second': 1.133, 'eval_steps_per_second': 0.142, 'epoch': 2.0}
Training Time: 12029.70 sec
Trainable Params: 296450


In [ ]:
# ==============================
# FIX: Upgrade torchao to fix ImportError
# ==============================
!pip install torchao>=0.16.0